# Tiny robot language model: training notebook

In [2]:
import os, glob

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.chdir(WORK) 


def find_data_dir():
    for root in ["/kaggle/input", os.path.join(WORK, "data")]:
        hits = sorted(glob.glob(os.path.join(root, "**", "train.jsonl"), recursive=True))
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError("Couldn't find train.jsonl.")


DATA_DIR = find_data_dir()
print("data folder:", DATA_DIR, "->", sorted(os.listdir(DATA_DIR)))

MODEL_CFG = dict(d_model=256, n_layer=4, n_head=4, d_ff=1024, max_len=256, dropout=0.1)
TRAIN_CFG = dict(epochs=30, batch_size=128, lr=1e-3, weight_decay=0.1, warmup_steps=200,
                 grad_clip=1.0, val_fraction=0.02, val_decode_max=600, seed=42)
MAX_NEW_TOKENS = 48

data folder: /kaggle/input/datasets/felixculpa/robotdata -> ['test.jsonl', 'train.jsonl']


In [5]:
%%writefile tiny_lm_numpy.py
#!/usr/bin/env python3
import json
import re
import sys
import time
import numpy as np

FANCY = {"\u2019": "'", "\u2018": "'", "\u201c": '"', "\u201d": '"',
         "\u2013": "-", "\u2014": "-", "\u2026": "..."}
INPUT_CHARS = "abcdefghijklmnopqrstuvwxyz0123456789 .,!?'-:;()/&\""
ALLOWED_CHARS = set(INPUT_CHARS)
MAX_INPUT_CHARS = 200


def normalize_input(text):
    """Lowercase, plain ASCII, allowed characters only, single spaces."""
    for k, v in FANCY.items():
        text = text.replace(k, v)
    text = text.lower()
    text = "".join(ch if ch in ALLOWED_CHARS else " " for ch in text)
    return re.sub(r"\s+", " ", text).strip()


SPECIALS = ["<pad>", "<bos>", "<sep>", "<eos>", "<unk>"]
OUTPUT_WORDS = ["FWD", "BACK", "LEFT", "RIGHT", "UNTIL", "SEARCH", "WATCH", "STOP",
                "UNKNOWN", "bottle", "cup", "chair", "person"]
VOCAB = SPECIALS + list(INPUT_CHARS) + OUTPUT_WORDS
TOK2ID = {t: i for i, t in enumerate(VOCAB)}
PAD, BOS, SEP, EOS, UNK = [TOK2ID[t] for t in SPECIALS]
OUTPUT_SYMBOLS = OUTPUT_WORDS + list("0123456789") + [".", "s", ";"]
OUTPUT_IDS = sorted(TOK2ID[t] for t in OUTPUT_SYMBOLS) + [EOS]
NUMBER_PARTS = set("0123456789.s")

OUT_TOKEN_RE = re.compile(r"FWD|BACK|LEFT|RIGHT|UNTIL|SEARCH|WATCH|STOP|UNKNOWN|"
                          r"bottle|cup|chair|person|[0-9]|\.|s|;")


_OBJ = "bottle|cup|chair|person"
_STEP_RE = re.compile(r"^(?:(?:FWD|BACK|LEFT|RIGHT) (?:\d+(?:\.\d+)?s|\d+)|FWD UNTIL (?:%s)|SEARCH (?:%s))$"
                      % (_OBJ, _OBJ))
_WATCH_RE = re.compile(r"^WATCH (?:%s)$" % _OBJ)


def valid_output(o):
    if o in ("STOP", "UNKNOWN"):
        return True
    parts = o.split("; ")
    if _WATCH_RE.match(parts[0]):
        parts = parts[1:]
    return 1 <= len(parts) <= 4 and all(_STEP_RE.match(p) for p in parts)


def encode_input(text):
    text = normalize_input(text)
    return [BOS] + [TOK2ID.get(ch, UNK) for ch in text] + [SEP]


def encode_output(label):
    toks = OUT_TOKEN_RE.findall(label)
    if decode_tokens(toks) != label:
        raise ValueError("label does not round-trip through the tokenizer: %r" % label)
    return [TOK2ID[t] for t in toks]


def decode_tokens(toks):
    pieces = []
    for t in toks:
        if t in NUMBER_PARTS and pieces and pieces[-1][0].isdigit() and t != ";":
            pieces[-1] += t          # continue a number like 1.5s
        else:
            pieces.append(t)
    return " ".join(pieces).replace(" ;", ";")


def decode_output(ids):
    return decode_tokens([VOCAB[i] for i in ids])


# Model
def _layer_norm(x, w, b, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = ((x - mu) ** 2).mean(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps) * w + b


def _gelu_tanh(x):
    return 0.5 * x * (1.0 + np.tanh(np.float32(0.7978845608028654) * (x + np.float32(0.044715) * x ** 3)))


class TinyLMNumpy(object):
    def __init__(self, weights_path="tiny_lm_weights.npz", config_path="tiny_lm_config.json"):
        with open(config_path) as f:
            cfg = json.load(f)
        if cfg["vocab"] != VOCAB:
            raise ValueError("vocabulary in config does not match this file: use the matching "
                             "tiny_lm_numpy.py from the same training run")
        self.d = cfg["d_model"]
        self.n_layer = cfg["n_layer"]
        self.n_head = cfg["n_head"]
        self.hd = self.d // self.n_head
        self.max_len = cfg["max_len"]
        self.max_new = cfg.get("max_new_tokens", 48)
        w = np.load(weights_path, allow_pickle=False)
        g = lambda k: np.ascontiguousarray(w[k].astype(np.float32))
        self.tok = g("tok_emb.weight")                 # (V, d), also the output layer
        self.pos = g("pos_emb.weight")                 # (max_len, d)
        self.layers = []
        for i in range(self.n_layer):
            p = "blocks.%d." % i
            self.layers.append({
                "ln1_w": g(p + "ln1.weight"), "ln1_b": g(p + "ln1.bias"),
                "qkv_w": np.ascontiguousarray(g(p + "attn.qkv.weight").T), "qkv_b": g(p + "attn.qkv.bias"),
                "proj_w": np.ascontiguousarray(g(p + "attn.proj.weight").T), "proj_b": g(p + "attn.proj.bias"),
                "ln2_w": g(p + "ln2.weight"), "ln2_b": g(p + "ln2.bias"),
                "fc1_w": np.ascontiguousarray(g(p + "mlp.fc1.weight").T), "fc1_b": g(p + "mlp.fc1.bias"),
                "fc2_w": np.ascontiguousarray(g(p + "mlp.fc2.weight").T), "fc2_b": g(p + "mlp.fc2.bias"),
            })
        self.lnf_w, self.lnf_b = g("ln_f.weight"), g("ln_f.bias")
        self.scale = np.float32(1.0 / np.sqrt(self.hd))
        self.out_mask = np.full(len(VOCAB), -np.inf, dtype=np.float32)
        self.out_mask[OUTPUT_IDS] = 0.0

    def _forward(self, ids, cache):
        """Run new token ids through the model, extending the cache. Returns last-position logits."""
        n, past = len(ids), cache["len"]
        if past + n > self.max_len:
            raise ValueError("sequence longer than max_len")
        x = self.tok[ids] + self.pos[past:past + n]                       # (n, d)
        mask = None
        if n > 1:
            mask = np.triu(np.ones((n, past + n), dtype=bool), k=past + 1)
        for i, L in enumerate(self.layers):
            h = _layer_norm(x, L["ln1_w"], L["ln1_b"])
            qkv = h @ L["qkv_w"] + L["qkv_b"]                              # (n, 3d)
            q, k, v = [a.reshape(n, self.n_head, self.hd).transpose(1, 0, 2)
                       for a in (qkv[:, :self.d], qkv[:, self.d:2 * self.d], qkv[:, 2 * self.d:])]
            if past:
                k = np.concatenate([cache["k"][i], k], axis=1)
                v = np.concatenate([cache["v"][i], v], axis=1)
            if i < len(cache["k"]):
                cache["k"][i], cache["v"][i] = k, v
            else:
                cache["k"].append(k)
                cache["v"].append(v)
            att = (q @ k.transpose(0, 2, 1)) * self.scale                  # (H, n, past+n)
            if mask is not None:
                att = np.where(mask[None, :, :], -np.inf, att)
            att = att - att.max(axis=-1, keepdims=True)
            att = np.exp(att)
            att /= att.sum(axis=-1, keepdims=True)
            y = (att @ v).transpose(1, 0, 2).reshape(n, self.d)
            x = x + y @ L["proj_w"] + L["proj_b"]
            h = _layer_norm(x, L["ln2_w"], L["ln2_b"])
            x = x + _gelu_tanh(h @ L["fc1_w"] + L["fc1_b"]) @ L["fc2_w"] + L["fc2_b"]
        cache["len"] = past + n
        last = _layer_norm(x[-1], self.lnf_w, self.lnf_b)
        return last @ self.tok.T                                            # (V,)

    def generate_ids(self, prompt_ids):
        cache = {"len": 0, "k": [], "v": []}
        logits = self._forward(prompt_ids, cache)
        out = []
        for _ in range(self.max_new):
            t = int(np.argmax(logits + self.out_mask))
            if t == EOS:
                break
            out.append(t)
            if cache["len"] + 1 > self.max_len:
                break
            logits = self._forward([t], cache)
        return out

    def predict(self, text):
        t0 = time.perf_counter()
        norm = normalize_input(text)
        info = {"normalized": norm, "raw": "", "valid": True}
        if not norm or len(norm) > MAX_INPUT_CHARS:
            info["valid"] = False
            info["ms"] = (time.perf_counter() - t0) * 1000.0
            return "UNKNOWN", info
        raw = decode_output(self.generate_ids(encode_input(norm)))
        info["raw"] = raw
        info["valid"] = valid_output(raw)
        info["ms"] = (time.perf_counter() - t0) * 1000.0
        return (raw if info["valid"] else "UNKNOWN"), info


def _main():
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", default="tiny_lm_weights.npz")
    ap.add_argument("--config", default="tiny_lm_config.json")
    ap.add_argument("--bench", action="store_true", help="time 20 typical commands")
    a = ap.parse_args()
    lm = TinyLMNumpy(a.weights, a.config)
    if a.bench:
        cmds = ["go forward 5 sec", "turn left", "go forward until you see a bottle",
                "go forward 5 sec then turn left 3 sec and stop if you see a bottle",
                "back up a little", "find the cup", "stop", "make me coffee",
                "drive 2 meters, turn right 90 degrees, then search for a bottle",
                "turn around"] * 2
        lm.predict("warm up")
        times = [lm.predict(c)[1]["ms"] for c in cmds]
        print("average %.1f ms, slowest %.1f ms over %d commands" % (sum(times) / len(times), max(times), len(times)))
        return
    print("Type a command (empty line to quit).")
    while True:
        try:
            text = input("> ")
        except EOFError:
            break
        if not text.strip():
            break
        codes, info = lm.predict(text)
        print("  %s   (%.1f ms%s)" % (codes, info["ms"], "" if info["valid"] else ", model output invalid: " + repr(info["raw"])))


if __name__ == "__main__":
    _main()

Overwriting tiny_lm_numpy.py


In [6]:
import importlib, json, math, random, sys, time
sys.path.insert(0, WORK)  # so tiny_lm_numpy.py (just written) can be imported
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiny_lm_numpy as T
importlib.reload(T)

random.seed(TRAIN_CFG["seed"]); np.random.seed(TRAIN_CFG["seed"]); torch.manual_seed(TRAIN_CFG["seed"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU: training will be slow")
print("torch", torch.__version__, "| vocabulary size:", len(T.VOCAB))

device: cuda | Tesla T4
torch 2.11.0+cu128 | vocabulary size: 68


In [7]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


all_train = load_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
tests = {} 
for p in sorted(glob.glob(os.path.join(DATA_DIR, "test*.jsonl"))):
    tests[os.path.splitext(os.path.basename(p))[0]] = load_jsonl(p)
print("train rows:", len(all_train), "| test sets:", {k: len(v) for k, v in tests.items()})


def encode_row(row):
    prompt = T.encode_input(row["input"])
    ids = prompt + T.encode_output(row["output"]) + [T.EOS]
    return ids, len(prompt)


problems = []
for name, rows in [("train", all_train)] + list(tests.items()):
    for r in rows:
        try:
            if not T.valid_output(r["output"]):
                problems.append((name, "invalid label", r)); continue
            ids, _ = encode_row(r)
            if len(ids) > MODEL_CFG["max_len"]:
                problems.append((name, "too long", r))
        except (ValueError, KeyError) as e:
            problems.append((name, repr(e), r))
print("problems found:", len(problems))
for p in problems[:10]:
    print("   ", p)
assert not problems, "Fix these rows in the data before training."

rng = random.Random(TRAIN_CFG["seed"])
shuffled = all_train[:]
rng.shuffle(shuffled)
n_val = max(200, int(len(shuffled) * TRAIN_CFG["val_fraction"]))
val_rows, train_rows = shuffled[:n_val], shuffled[n_val:]
train_enc = [encode_row(r) for r in train_rows]
val_enc = [encode_row(r) for r in val_rows]
lens = [len(ids) for ids, _ in train_enc]
print("train %d | validation %d | tokens per example: avg %.0f, max %d"
      % (len(train_enc), len(val_enc), sum(lens) / len(lens), max(lens)))
print("example:", train_rows[0]["input"], "->", train_rows[0]["output"])

train rows: 41053 | test sets: {'test': 3934}
problems found: 0
train 40232 | validation 821 | tokens per example: avg 70, max 199
example: hey robot um drive forward until you can see a door -> UNKNOWN


In [8]:
class SelfAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head, self.d = c["n_head"], c["d_model"]
        self.qkv = nn.Linear(self.d, 3 * self.d)
        self.proj = nn.Linear(self.d, self.d)
        self.p = c["dropout"]
        self.resid_drop = nn.Dropout(c["dropout"])

    def forward(self, x):
        B, L, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, L, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, L, C)
        return self.resid_drop(self.proj(y))


class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.fc1 = nn.Linear(c["d_model"], c["d_ff"])
        self.act = nn.GELU(approximate="tanh")  # same formula as the NumPy runner
        self.fc2 = nn.Linear(c["d_ff"], c["d_model"])
        self.drop = nn.Dropout(c["dropout"])

    def forward(self, x):
        return self.drop(self.fc2(self.act(self.fc1(x))))


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(c["d_model"]), SelfAttention(c)
        self.ln2, self.mlp = nn.LayerNorm(c["d_model"]), MLP(c)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class TinyLM(nn.Module):
    def __init__(self, c, vocab_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, c["d_model"])
        self.pos_emb = nn.Embedding(c["max_len"], c["d_model"])
        self.drop = nn.Dropout(c["dropout"])
        self.blocks = nn.ModuleList([Block(c) for _ in range(c["n_layer"])])
        self.ln_f = nn.LayerNorm(c["d_model"])
        self.head = nn.Linear(c["d_model"], vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # tied: the output layer reuses the embedding
        self.apply(self._init)
        for n, p in self.named_parameters():
            if n.endswith("proj.weight") or n.endswith("fc2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * c["n_layer"]))

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        pos = torch.arange(idx.size(1), device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss


model = TinyLM(MODEL_CFG, len(T.VOCAB)).to(DEVICE)
N_PARAMS = sum(p.numel() for p in model.parameters())
print("parameters: %d (%.2fM)" % (N_PARAMS, N_PARAMS / 1e6))

parameters: 3242496 (3.24M)


In [9]:
OUT_MASK = torch.full((len(T.VOCAB),), float("-inf"))
OUT_MASK[T.OUTPUT_IDS] = 0.0
OUT_MASK = OUT_MASK.to(DEVICE)


def make_batch(items):
    L = max(len(ids) for ids, _ in items)
    x = torch.full((len(items), L - 1), T.PAD, dtype=torch.long)
    y = torch.full((len(items), L - 1), -100, dtype=torch.long)
    for b, (ids, plen) in enumerate(items):
        t = torch.tensor(ids, dtype=torch.long)
        x[b, :len(ids) - 1] = t[:-1]
        y[b, plen - 1:len(ids) - 1] = t[plen:]  # targets: answer tokens + <eos> only
    return x.to(DEVICE), y.to(DEVICE)


@torch.no_grad()
def greedy_same_length(model, prompts):
    """Greedy decoding for prompts that all have the same length (so no padding is needed)."""
    x = torch.tensor(prompts, dtype=torch.long, device=DEVICE)
    outs, done = [[] for _ in prompts], [False] * len(prompts)
    for _ in range(MAX_NEW_TOKENS):
        logits, _ = model(x)
        nxt = (logits[:, -1, :] + OUT_MASK).argmax(-1)
        for i, t in enumerate(nxt.tolist()):
            if not done[i]:
                if t == T.EOS:
                    done[i] = True
                else:
                    outs[i].append(t)
        if all(done) or x.size(1) + 1 > MODEL_CFG["max_len"]:
            break
        x = torch.cat([x, nxt[:, None]], dim=1)
    return outs


def predict_texts(model, texts, batch_size=256):
    """Same rules as the robot: normalize, decode, and turn invalid output into UNKNOWN."""
    model.eval()
    preds, groups = [None] * len(texts), {}
    for i, text in enumerate(texts):
        norm = T.normalize_input(text)
        if not norm or len(norm) > T.MAX_INPUT_CHARS:
            preds[i] = "UNKNOWN"
            continue
        p = T.encode_input(norm)
        groups.setdefault(len(p), []).append((i, p))
    for _, items in sorted(groups.items()):
        for s in range(0, len(items), batch_size):
            chunk = items[s:s + batch_size]
            for (i, _), o in zip(chunk, greedy_same_length(model, [p for _, p in chunk])):
                raw = T.decode_output(o)
                preds[i] = raw if T.valid_output(raw) else "UNKNOWN"
    return preds


def evaluate(model, rows, name="", show_failures=0):
    preds = predict_texts(model, [r["input"] for r in rows])
    correct = [p == r["output"] for p, r in zip(preds, rows)]
    wrong = [(r, p) for r, p, c in zip(rows, preds, correct) if not c]
    refused = sum(1 for _, p in wrong if p == "UNKNOWN")
    by_kind = {}
    for r, c in zip(rows, correct):
        a = by_kind.setdefault(r.get("kind", "all"), [0, 0])
        a[0] += c
        a[1] += 1
    res = {"n": len(rows), "exact_match": sum(correct) / max(1, len(rows)),
           "wrong_but_refused": refused, "wrong_not_refused": len(wrong) - refused,
           "by_kind": {k: round(a[0] / a[1], 4) for k, a in sorted(by_kind.items())}}
    if name:
        print("%-13s exact match %.2f%% (%d/%d) | wrong but refused (safe): %d | wrong, not refused: %d"
              % (name, 100 * res["exact_match"], sum(correct), len(rows), refused, len(wrong) - refused))
        print("              by kind:", res["by_kind"])
        for r, p in wrong[:show_failures]:
            print("     %-58s want %-30s got %s" % (r["input"][:58], r["output"], p))
    return res


@torch.no_grad()
def val_loss(model):
    model.eval()
    total, count = 0.0, 0
    for s in range(0, len(val_enc), 256):
        x, y = make_batch(val_enc[s:s + 256])
        _, loss = model(x, y)
        n = (y != -100).sum().item()
        total += loss.item() * n
        count += n
    return total / count

In [ ]:
decay = [p for n, p in model.named_parameters() if p.dim() >= 2]
no_decay = [p for n, p in model.named_parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": TRAIN_CFG["weight_decay"]},
                         {"params": no_decay, "weight_decay": 0.0}],
                        lr=TRAIN_CFG["lr"], betas=(0.9, 0.95))
steps_per_epoch = math.ceil(len(train_enc) / TRAIN_CFG["batch_size"])
total_steps = steps_per_epoch * TRAIN_CFG["epochs"]


def lr_factor(step):  # warm up, then cosine decay to 10% of the peak
    if step < TRAIN_CFG["warmup_steps"]:
        return (step + 1) / TRAIN_CFG["warmup_steps"]
    prog = (step - TRAIN_CFG["warmup_steps"]) / max(1, total_steps - TRAIN_CFG["warmup_steps"])
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))


sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
val_check = val_rows[:TRAIN_CFG["val_decode_max"]]
best = {"acc": -1.0, "loss": float("inf"), "epoch": 0}
history, t0 = [], time.time()
for epoch in range(1, TRAIN_CFG["epochs"] + 1):
    model.train()
    order = list(range(len(train_enc)))
    random.shuffle(order)
    run_loss = 0.0
    for s in range(0, len(order), TRAIN_CFG["batch_size"]):
        x, y = make_batch([train_enc[i] for i in order[s:s + TRAIN_CFG["batch_size"]]])
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CFG["grad_clip"])
        opt.step()
        sched.step()
        run_loss += loss.item()
    vl = val_loss(model)
    va = evaluate(model, val_check)["exact_match"]
    history.append({"epoch": epoch, "train_loss": run_loss / steps_per_epoch, "val_loss": vl, "val_exact": va})
    better = va > best["acc"] or (va == best["acc"] and vl < best["loss"])
    if better:
        best = {"acc": va, "loss": vl, "epoch": epoch}
        torch.save(model.state_dict(), os.path.join(WORK, "best.pt"))
    print("epoch %2d | train loss %.4f | val loss %.4f | val exact %.2f%% | %4.0fs%s"
          % (epoch, run_loss / steps_per_epoch, vl, 100 * va, time.time() - t0, "  <- saved" if better else ""))
print("best epoch:", best)

epoch  1 | train loss 1.1416 | val loss 0.6778 | val exact 21.67% |   42s  <- saved
epoch  2 | train loss 0.6146 | val loss 0.4770 | val exact 38.33% |   85s  <- saved
epoch  3 | train loss 0.4269 | val loss 0.3372 | val exact 49.50% |  130s  <- saved
epoch  4 | train loss 0.3023 | val loss 0.2035 | val exact 65.50% |  175s  <- saved
epoch  5 | train loss 0.2275 | val loss 0.1498 | val exact 74.83% |  220s  <- saved
epoch  6 | train loss 0.1736 | val loss 0.1285 | val exact 76.67% |  265s  <- saved
epoch  7 | train loss 0.1395 | val loss 0.1054 | val exact 84.17% |  310s  <- saved
epoch  8 | train loss 0.1150 | val loss 0.0792 | val exact 88.33% |  355s  <- saved
epoch  9 | train loss 0.0954 | val loss 0.0650 | val exact 88.17% |  400s
epoch 10 | train loss 0.0812 | val loss 0.0432 | val exact 92.33% |  445s  <- saved
epoch 11 | train loss 0.0696 | val loss 0.0403 | val exact 92.00% |  490s
epoch 12 | train loss 0.0594 | val loss 0.0381 | val exact 91.83% |  535s
epoch 13 | train loss 

In [9]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ep = [h["epoch"] for h in history]
    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(ep, [h["train_loss"] for h in history], label="train loss")
    ax1.plot(ep, [h["val_loss"] for h in history], label="validation loss")
    ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.legend(loc="upper left")
    ax2 = ax1.twinx()
    ax2.plot(ep, [100 * h["val_exact"] for h in history], "g--", label="validation exact match")
    ax2.set_ylabel("exact match (%)"); ax2.legend(loc="center right")
    plt.title("Tiny robot LM training (%.1fM parameters)" % (N_PARAMS / 1e6))
    plt.tight_layout(); plt.savefig(os.path.join(WORK, "loss_curve.png"), dpi=150)
    print("saved loss_curve.png")
except Exception as e:
    print("plot skipped:", e)

saved loss_curve.png


In [10]:
model.load_state_dict(torch.load(os.path.join(WORK, "best.pt"), map_location=DEVICE))
model.eval()
results = {"n_params": N_PARAMS, "best_epoch": best["epoch"], "history": history, "tests": {}}
for name, rows in tests.items():
    results["tests"][name] = evaluate(model, rows, name, show_failures=12)
    print()
with open(os.path.join(WORK, "results.json"), "w") as f:
    json.dump(results, f, indent=1)
print("saved results.json")

test          exact match 94.38% (3713/3934) | wrong but refused (safe): 4 | wrong, not refused: 217
              by kind: {'all': 0.9438}
     rotate right by 90deg, turn to opposite now afterwards ok  want RIGHT 90; LEFT 180; FWD 2s; LEFT 270 got RIGHT 90; LEFT 180; LEFT 270
     scoot forward now next and then could you do a rear facing want FWD 2s; LEFT 180               got FWD 2s; LEFT 90
     if you spot a plastic bottle stop and then back up, come o want WATCH bottle; BACK 2s; FWD 2s  got WATCH bottle; BACK 2s; FWD 2s; LEFT 90
     make a turn, thn ok turn later then go onward and fwd slig want LEFT 90; LEFT 90; FWD 2s; FWD 1s got LEFT 90; LEFT 90; FWD 1s
     go until a chair immediately after that ok now back up in  want FWD UNTIL chair; BACK 2s; FWD 2s; FWD 400 got FWD UNTIL chair; BACK 2s; FWD 400
     on sight of an office chair stop and then, could you pleas want WATCH chair; LEFT 90; LEFT 180 got WATCH chair; LEFT 90; LEFT 90; LEFT 180
     can you go right and then twi

In [11]:
weights = {k: v.detach().float().cpu().numpy() for k, v in model.state_dict().items() if k != "head.weight"}
np.savez(os.path.join(WORK, "tiny_lm_weights.npz"), **weights)
config = dict(MODEL_CFG, vocab=T.VOCAB, max_new_tokens=MAX_NEW_TOKENS, n_params=N_PARAMS)
with open(os.path.join(WORK, "tiny_lm_config.json"), "w") as f:
    json.dump(config, f, indent=1)
for fn in ["tiny_lm_weights.npz", "tiny_lm_config.json", "tiny_lm_numpy.py"]:
    print("%-22s %8.2f MB" % (fn, os.path.getsize(os.path.join(WORK, fn)) / 1e6))

tiny_lm_weights.npz       12.98 MB
tiny_lm_config.json        0.00 MB
tiny_lm_numpy.py           0.01 MB
